In [1]:
pip install pandas yfinance pandas_ta



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import yfinance as yf
import numpy as np

symbol = "TITAN.NS"
df = yf.download(symbol, start="2024-01-01")

# True Range
df['TR'] = np.maximum(
    df['High'] - df['Low'],
    np.maximum(
        abs(df['High'] - df['Close'].shift(1)),
        abs(df['Low'] - df['Close'].shift(1))
    )
)

# Directional Movement
df['+DM'] = np.where(
    (df['High'] - df['High'].shift(1)) > (df['Low'].shift(1) - df['Low']),
    np.maximum(df['High'] - df['High'].shift(1), 0),
    0
)

df['-DM'] = np.where(
    (df['Low'].shift(1) - df['Low']) > (df['High'] - df['High'].shift(1)),
    np.maximum(df['Low'].shift(1) - df['Low'], 0),
    0
)

# Wilder smoothing
period = 14
df['TR_smooth'] = df['TR'].rolling(period).mean()
df['+DM_smooth'] = df['+DM'].rolling(period).mean()
df['-DM_smooth'] = df['-DM'].rolling(period).mean()

# DI values
df['+DI'] = 100 * (df['+DM_smooth'] / df['TR_smooth'])
df['-DI'] = 100 * (df['-DM_smooth'] / df['TR_smooth'])

# DX & ADX
df['DX'] = 100 * abs(df['+DI'] - df['-DI']) / (df['+DI'] + df['-DI'])
df['ADX'] = df['DX'].rolling(period).mean()

print(df[['Close', '+DI', '-DI', 'ADX']].tail())



/tmp/ipykernel_192319/903586252.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start="2024-01-01")
[*********************100%***********************]  1 of 1 completed

Price             Close        +DI        -DI        ADX
Ticker         TITAN.NS                                 
Date                                                    
2025-12-18  3919.300049  17.179412  22.260626  25.289897
2025-12-19  3930.100098  21.558684  18.215358  23.834560
2025-12-22  3933.800049  21.518993  18.181823  23.639940
2025-12-23  3925.800049  23.794816   9.443749  25.440240
2025-12-24  3909.300049  24.305563  10.719688  27.103433


In [3]:
df = df.dropna().reset_index(drop=True)

adx = df['ADX'].iloc[-1]
plus_di = df['+DI'].iloc[-1]
minus_di = df['-DI'].iloc[-1]


In [4]:
adx = float(adx)
plus_di = float(plus_di)
minus_di = float(minus_di)


In [5]:
if adx > 25:
    if plus_di > minus_di:
        print("📈 Strong Uptrend → Buy / Hold")
    else:
        print("📉 Strong Downtrend → Sell / Short")
else:
    print("⚠️ Sideways Market → Avoid")


📈 Strong Uptrend → Buy / Hold
